## Configuration - Define all paths here

In [2]:
# ============================================================================
# CONFIGURATION - Edit these paths according to your environment
# ============================================================================
import os 
# Base directory for all data (adjust this to your environment)
BASE_DIR = "./data"

# Database path
xPath_DB = os.path.join(BASE_DIR, "db")
xDB = os.path.join(xPath_DB, "grants_datasets.db")
xDBCon = f"sqlite:///{xDB}"

# Downloaded datasets path
xPath_downloaded_datasets = os.path.join(BASE_DIR, "downloaded")
xPath_dsets_EU = os.path.join(xPath_downloaded_datasets, "01_EU_FP", "01_projects")

# Publications data path (if available separately)
xPath_publications = os.path.join(BASE_DIR, "publications")

# Create directories if they don't exist
os.makedirs(xPath_DB, exist_ok=True)
os.makedirs(xPath_dsets_EU, exist_ok=True)

print("Configuration loaded:")
print(f"  BASE_DIR: {BASE_DIR}")
print(f"  Database: {xDB}")
print(f"  Datasets: {xPath_dsets_EU}")
print(f"  Publications: {xPath_publications}")

Configuration loaded:
  BASE_DIR: ./data
  Database: ./data/db/grants_datasets.db
  Datasets: ./data/downloaded/01_EU_FP/01_projects
  Publications: ./data/publications


## EU Funding Data 

## STEP 1 : download the datasets 


#### project data #### 

Run these commands in your terminal to download the data:

```bash
# Create directories
mkdir -p data/downloaded/01_EU_FP/01_projects
mkdir -p data/publications

# FP1-FP6 project data
cd data/downloaded/01_EU_FP/01_projects
wget https://cordis.europa.eu/data/FP1/cordis-fp1projects.xlsx
wget https://cordis.europa.eu/data/FP1/cordis-fp1organizations.xlsx
wget https://cordis.europa.eu/data/FP2/cordis-fp2projects.xlsx
wget https://cordis.europa.eu/data/FP2/cordis-fp2organizations.xlsx
wget https://cordis.europa.eu/data/FP3/cordis-fp3projects.xlsx
wget https://cordis.europa.eu/data/FP3/cordis-fp3organizations.xlsx
wget https://cordis.europa.eu/data/FP4/cordis-fp4projects.xlsx
wget https://cordis.europa.eu/data/FP4/cordis-fp4organizations.xlsx
wget https://cordis.europa.eu/data/FP5/cordis-fp5projects.xlsx
wget https://cordis.europa.eu/data/FP5/cordis-fp5organizations.xlsx
wget https://cordis.europa.eu/data/FP6/cordis-fp6projects.xlsx
wget https://cordis.europa.eu/data/FP6/cordis-fp6organizations.xlsx

# FP7, H2020, Horizon Europe (ZIP files)
wget https://cordis.europa.eu/data/cordis-fp7projects-xlsx.zip
wget https://cordis.europa.eu/data/cordis-h2020projects-xlsx.zip
wget https://cordis.europa.eu/data/cordis-HORIZONprojects-xlsx.zip

# Programme reference data
wget https://cordis.europa.eu/data/reference/cordisref-FP6programmes.csv
wget https://cordis.europa.eu/data/reference/cordisref-FP7programmes.csv
wget https://cordis.europa.eu/data/reference/cordisref-H2020programmes-xlsx.zip
wget https://cordis.europa.eu/data/reference/cordisref-HORIZONprogrammes-xlsx.zip

# ERC Principal Investigators
wget https://cordis.europa.eu/data/cordis-fp7projects-xml.zip
wget https://cordis.europa.eu/data/cordis-h2020-erc-pi.xlsx

# Reference data
wget https://cordis.europa.eu/data/reference/cordisref-countries.csv
```


### STEP 2 : save into a database 

In [4]:
import os 
import inflection as infl  
import pandas as pd 

import zipfile
import xmltodict
import warnings
warnings.simplefilter("ignore")

####  FP1 to FP6 are single excel files each  ### 

In [6]:
xFile_lst = ['cordis-fp1projects.xlsx', 'cordis-fp2projects.xlsx', 
             'cordis-fp3projects.xlsx', 'cordis-fp4projects.xlsx', 
             'cordis-fp5projects.xlsx', 'cordis-fp6projects.xlsx',
             'cordis-fp1organizations.xlsx', 'cordis-fp2organizations.xlsx', 
             'cordis-fp3organizations.xlsx', 'cordis-fp4organizations.xlsx',             
             'cordis-fp5organizations.xlsx', 'cordis-fp6organizations.xlsx'        
            ]

for xFile in xFile_lst:
    xFile_name = os.path.join(xPath_dsets_EU , xFile)
    
    #name of the table in sqlite 
    xtab_name = xFile.replace('-', '_').replace('.xlsx', '')
    #dataframe 
    df = pd.read_excel(xFile_name)
    # change the names of the columns from Camelcas to under_score 
    xcols = [infl.underscore(x) for x in df.columns]
    df.columns = xcols
    
    #Save 
    df.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 
    
    print('saved:', xtab_name)
    

print('OK')

FileNotFoundError: [Errno 2] No such file or directory: './data/downloaded/01_EU_FP/01_projects/cordis-fp1projects.xlsx'

#### FP7 #### 

In [8]:
xFile = 'cordis-fp7projects-xlsx.zip'

xfile_zip = os.path.join(xPath_dsets_EU , xFile)

with zipfile.ZipFile(xfile_zip) as zf:
    xtabname_prefix = xfile_zip.split('/')[-1:][0].split('.')[0].replace('-', '_').replace('xlsx', '')
    

    for xfile in zf.filelist:
        xfile_name = xfile.filename
        
        if xfile_name.endswith('.xlsx'):
            xtab_name_suffix = xfile.filename.replace('/', '_').replace('.xlsx', '')
            xtab_name = xtabname_prefix  + infl.underscore(xtab_name_suffix)
            xtab_name = xtab_name.replace('_xlsx_', '_')
        
            with zf.open(xfile_name) as myZip:
                df = pd.read_excel(myZip, engine="openpyxl")
                xcols = [infl.underscore(x) for x in df.columns]
                df.columns = xcols
                #save 
                df.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 

                print('saved:', xtab_name, '---records:', len(df))
            
            
            
            
print('OK')

saved: cordis_fp7projects_web_link ---records: 10792
saved: cordis_fp7projects_legal_basis ---records: 25785
saved: cordis_fp7projects_topics ---records: 26153
saved: cordis_fp7projects_organization ---records: 140064
saved: cordis_fp7projects_web_item ---records: 11601
saved: cordis_fp7projects_euro_sci_voc ---records: 66986
saved: cordis_fp7projects_project ---records: 25785
OK


#### H2020  #### 

In [9]:

xFile = 'cordis-h2020projects-xlsx.zip'

xfile_zip = os.path.join(xPath_dsets_EU , xFile)


with zipfile.ZipFile(xfile_zip) as zf:
    xtabname_prefix = xfile_zip.split('/')[-1:][0].split('.')[0].replace('-', '_').replace('xlsx', '')
    

    for xfile in zf.filelist:
        xfile_name = xfile.filename
        
        if xfile_name.endswith('.xlsx'):
            xtab_name_suffix = xfile.filename.replace('/', '_').replace('.xlsx', '')
            xtab_name = xtabname_prefix  + infl.underscore(xtab_name_suffix)
            xtab_name = xtab_name.replace('_xlsx_', '_')            
        
            with zf.open(xfile_name) as myZip:
                df = pd.read_excel(myZip, engine="openpyxl")
                xcols = [infl.underscore(x) for x in df.columns]
                df.columns = xcols
                #save 
                df.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 

                print('saved:', xtab_name, '---records:', len(df))
            
            
            
            
print('OK')

saved: cordis_h2020projects_web_link ---records: 229123
saved: cordis_h2020projects_legal_basis ---records: 65812
saved: cordis_h2020projects_topics ---records: 35389
saved: cordis_h2020projects_organization ---records: 178783
saved: cordis_h2020projects_web_item ---records: 11
saved: cordis_h2020projects_euro_sci_voc ---records: 112222
saved: cordis_h2020projects_project ---records: 35389
OK


#### Horizon Europe  #### 

In [10]:
xFile = 'cordis-HORIZONprojects-xlsx.zip'

xfile_zip = os.path.join(xPath_dsets_EU , xFile)

with zipfile.ZipFile(xfile_zip) as zf:
    xtabname_prefix = xfile_zip.split('/')[-1:][0].split('.')[0].replace('-', '_').replace('xlsx', '')
    

    for xfile in zf.filelist:
        xfile_name = xfile.filename
        
        if xfile_name.endswith('.xlsx'):
            xtab_name_suffix = xfile.filename.replace('/', '_').replace('.xlsx', '')
            xtab_name = xtabname_prefix  + infl.underscore(xtab_name_suffix)
            xtab_name = xtab_name.replace('_xlsx_', '_')  
        
            with zf.open(xfile_name) as myZip:
                df = pd.read_excel(myZip, engine="openpyxl")
                xcols = [infl.underscore(x) for x in df.columns]
                df.columns = xcols
                #save 
                df.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 

                print('saved:', xtab_name, '---records:', len(df))
            
            
            
            
print('OK')

saved: cordis_HORIZONprojects_policy_priorities ---records: 19644
saved: cordis_HORIZONprojects_web_link ---records: 43628
saved: cordis_HORIZONprojects_legal_basis ---records: 25551
saved: cordis_HORIZONprojects_topics ---records: 19706
saved: cordis_HORIZONprojects_organization ---records: 121723
saved: cordis_HORIZONprojects_web_item ---records: 6
saved: cordis_HORIZONprojects_euro_sci_voc ---records: 42003
saved: cordis_HORIZONprojects_project ---records: 19706
OK


## Principal Investigators - ERC only 

In [11]:
xlst_res = []

xFile = 'cordis-fp7projects-xml.zip'

xfile_zip = os.path.join(xPath_dsets_EU , xFile)


with zipfile.ZipFile(xfile_zip, 'r') as zip_ref:
    for file_name in zip_ref.namelist():
        if file_name.endswith('.xml'):
            with zip_ref.open(file_name) as file:
                xdata = file.read()
                xdict = xmltodict.parse(xdata).get('project')
                ## get project data 
                xproj_id = xdict.get('id') 
                xproj_rcn = xdict.get('rcn') 
                xproj_acronym = xdict.get('acronym')   
                
                xrels = xdict.get('relations')
                if not xrels:
                    continue                    
                    
                
                xorgs = xdict.get('relations').get('associations').get('organization')
                if isinstance(xorgs , dict):
                    xorgs = [xorgs ]
                    
                if not xorgs:
                    continue
                
                for xorg in xorgs:
                    #get organisations data 
                    xorg_id =  xorg.get('id')
                    xorg_rcn = xorg.get('rcn')
                    
                    if not xorg.get('relations'):
                        continue 
                    xassociations = xorg.get('relations').get('associations')
                    if not xassociations:
                        continue
                    

                    # get person data 
                    xperss = xassociations.get('person')
                    
                    if not xperss:
                        continue 
                    
                                      
                    if isinstance(xperss , dict):
                        xperss = [xperss]
                        

                    for xpers in xperss:
                        xpers_dict = {}
                        xpers_dict['proj_id'] = xproj_id
                        xpers_dict['proj_rcn'] = xproj_rcn 
                        xpers_dict['proj_acronym'] = xproj_acronym

                        xpers_dict['pers_type'] = xpers.get('@type')
                        xpers_dict['pi_rcn'] = xpers.get('rcn')                            
                        xpers_dict['pi_title'] = xpers.get('title')
                        xpers_dict['pi_name_first'] = xpers.get('firstNames')
                        xpers_dict['pi_name_last'] = xpers.get('lastName')   

                        xpers_dict['org_id'] = xorg_id
                        xpers_dict['org_rcn'] = xorg_rcn   

                        xlst_res.append(xpers_dict)
                
                
t1 = pd.DataFrame(xlst_res)
print(len(t1))

xtab_name = 'cordis_fp7_persons'
t1.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 
print('saved:', xtab_name, '---records:', len(t1))

138196
saved: cordis_fp7_persons ---records: 138196


In [12]:
## H2020 

xFile = os.path.join(xPath_dsets_EU , 'cordis-h2020-erc-pi.xlsx')

df = pd.read_excel(xFile)

xcols = [infl.underscore(x) for x in df.columns]
df.columns = xcols


xtab_name = 'cordis_h2020_persons'
df.to_sql(name = xtab_name,  con = xDBCon,  if_exists =  'replace', index = False) 
print('saved:', xtab_name, '---records:', len(df))

saved: cordis_h2020_persons ---records: 8043


## Results - publications 

Note: Publications data requires separate download from CORDIS.
Place the files in the publications directory and update the paths below.

In [ ]:
# Publications paths - update these when publications data is available
# Download from: https://cordis.europa.eu/data/

xl_fp7 = os.path.join(xPath_publications, 'FP7PC_DM_PROJ_PUBLICATIONS.xlsx')
xl_h2020_dir = os.path.join(xPath_publications, 'cordis-h2020projectPublications-xlsx', 'xlsx')
xl_he = os.path.join(xPath_publications, 'cordis-HORIZONprojectPublications-xlsx', 'xlsx', 'projectPublications.xlsx')

# Check if files exist
print("Publications data paths:")
print(f"  FP7: {xl_fp7} - exists: {os.path.exists(xl_fp7)}")
print(f"  H2020 dir: {xl_h2020_dir} - exists: {os.path.exists(xl_h2020_dir)}")
print(f"  HE: {xl_he} - exists: {os.path.exists(xl_he)}")

In [ ]:
# FP7 Publications (if available)
if os.path.exists(xl_fp7):
    dset_fp7 = pd.read_excel(xl_fp7)
    dset_fp7.columns = [x.lower() for x in dset_fp7.columns]
    dset_fp7.to_sql(name = 'cordis_publications_fp7', con = xDBCon, if_exists = 'replace', index = False) 
    print('saved FP7:', len(dset_fp7))
else:
    print("FP7 publications file not found. Skipping.")

In [ ]:
# H2020 Publications (if available)
if os.path.exists(xl_h2020_dir):
    xl_h2020_lst = [os.path.join(xl_h2020_dir, x) for x in os.listdir(xl_h2020_dir)]
    
    for xFile in xl_h2020_lst:
        xtab_name = 'cordis_publications_h2020_' + os.path.basename(xFile).replace('.xlsx', '')
        
        dset_h2020 = pd.read_excel(xFile)
        dset_h2020 = dset_h2020.astype(str)    
        dset_h2020.columns = [x.lower() for x in dset_h2020.columns]
        
        dset_h2020.to_sql(name = xtab_name, con = xDBCon, if_exists = 'replace', index = False)     
        
        print('saved:', xtab_name, len(dset_h2020))
else:
    print("H2020 publications directory not found. Skipping.")

In [ ]:
# Horizon Europe Publications (if available)
if os.path.exists(xl_he):
    dset_he = pd.read_excel(xl_he)
    dset_he = dset_he.astype(str)
    dset_he.columns = [x.lower() for x in dset_he.columns]
    dset_he.to_sql(name = 'cordis_publications_horizon_europe', con = xDBCon, if_exists = 'replace', index = False)     
    print('saved: cordis_publications_horizon_europe', len(dset_he))
else:
    print("Horizon Europe publications file not found. Skipping.")

In [ ]:
# Summary - list all tables in the database
import sqlite3
conn = sqlite3.connect(xDB)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
conn.close()

print("\n=== Database Summary ===")
print(f"Database: {xDB}")
print(f"Total tables: {len(tables)}")
print("\nTables:")
for table in sorted(tables):
    print(f"  - {table[0]}")